# Battery Prognostics - Quick Start Demo

End-to-end: Load → Validate → Features → Train → Evaluate → Safety

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from src.data.unified_loader import UnifiedDataLoader
from src.data.validator import DataValidator
from src.features.extractor import FeatureExtractor
from src.models import LSTMModel, GRUModel, TCNModel, TransformerModel, PINNModel
from src.uncertainty.scoring import compute_all_metrics
from src.safety.decision_engine import SafetyDecisionEngine

## 1. Load & Validate Data

In [ ]:
loader = UnifiedDataLoader()
df = loader.load_all(nasa_dir='../data/battery_data')
print(f'Loaded: {len(df)} rows, {df["battery_id"].nunique()} batteries')

validator = DataValidator()
df, report = validator.validate(df)
print(f'Validation: {report.pass_rate:.1%} pass rate')

## 2. Feature Extraction

In [ ]:
extractor = FeatureExtractor()
df = extractor.extract_all(df)
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in ('cycle', 'rul')]
df = df.dropna(subset=feature_cols + ['rul']).reset_index(drop=True)
print(f'{len(feature_cols)} features, {len(df)} rows')

## 3. Degradation Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for bat in df['battery_id'].unique():
    sub = df[df['battery_id'] == bat].sort_values('cycle')
    ax.plot(sub['cycle'], sub['capacity'], label=bat, linewidth=2)
ax.set_xlabel('Cycle'); ax.set_ylabel('Capacity (Ah)')
ax.legend(); ax.grid(alpha=0.3)
plt.title('Battery Capacity Degradation'); plt.show()

## 4. Train & Compare Models

In [ ]:
from src.data.splitter import DataSplitter

# Use B0018 as test
train_df = df[df['battery_id'] != 'B0018']
test_df = df[df['battery_id'] == 'B0018'].sort_values('cycle')

X_train = train_df[feature_cols].values
y_train = train_df['rul'].values
X_test = test_df[feature_cols].values
y_test = test_df['rul'].values

models = {
    'LSTM': LSTMModel(input_dim=len(feature_cols), hidden_dim=64, epochs=50, seq_length=20),
    'GRU': GRUModel(input_dim=len(feature_cols), hidden_dim=64, epochs=50, seq_length=20),
    'PINN': PINNModel(input_dim=len(feature_cols), hidden_dim=64, epochs=50),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    mean, lower, upper = model.predict(X_test)
    y_eval = y_test[-len(mean):]
    metrics = compute_all_metrics(y_eval, mean, lower, upper)
    results[name] = {'mean': mean, 'lower': lower, 'upper': upper, 'metrics': metrics}
    print(f'{name}: RMSE={metrics["RMSE"]:.2f}, CRPS={metrics["CRPS"]:.2f}, PICP={metrics["PICP"]:.2%}')

## 5. Prediction with Uncertainty

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 5), sharey=True)
cycles = test_df['cycle'].values
y_plot = y_test

for ax, (name, r) in zip(axes, results.items()):
    n = len(r['mean'])
    c = cycles[-n:]
    ax.plot(cycles, y_plot, 'k-', linewidth=2, label='Ground Truth')
    ax.plot(c, r['mean'], '--', linewidth=1.5, label='Prediction')
    ax.fill_between(c, r['lower'], r['upper'], alpha=0.2, label='95% CI')
    ax.set_title(f"{name} (RMSE={r['metrics']['RMSE']:.1f})")
    ax.set_xlabel('Cycle'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

axes[0].set_ylabel('RUL (cycles)')
plt.tight_layout(); plt.show()

## 6. Safety Decisions

In [ ]:
engine = SafetyDecisionEngine(rul_critical=10, rul_warning=30)
best = list(results.values())[0]
decisions = engine.decide_batch(best['mean'], best['lower'], best['upper'])

colors = {'GREEN': '#2ecc71', 'YELLOW': '#f39c12', 'RED': '#e74c3c'}
for d in decisions[-5:]:
    print(f'RUL={d.rul_estimate:.0f} [{d.confidence_lower:.0f},{d.confidence_upper:.0f}] → {d.level.value}: {d.action}')